# Milling Signal Analysis

Quick notebook for inspecting windows, FFT peaks, and physics-guided features. It defaults to the synthetic demo data; replace the loader call with a NASA data path when available.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from data_loader import load_milling_runs
from feature_extraction import extract_dataset_features
from physics_features import spindle_frequency_hz, tooth_passing_frequency_hz

runs = load_milling_runs(demo=True)
features = extract_dataset_features(runs, include_physics=True)
features.head()

In [ ]:
run = runs[-1]
signal = run.signals['vib_spindle'].to_numpy()
t = np.arange(signal.size) / run.sample_rate_hz

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[:1000], signal[:1000])
ax.set_title(f'Vibration window, VB={run.vb:.3f}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
plt.show()

In [ ]:
freqs = np.fft.rfftfreq(signal.size, d=1 / run.sample_rate_hz)
amp = np.abs(np.fft.rfft(signal - signal.mean()))
spindle = spindle_frequency_hz(run.speed_rpm)
tpf = tooth_passing_frequency_hz(run.speed_rpm, run.tooth_count)

fig, ax = plt.subplots(figsize=(10, 4))
mask = freqs < 500
ax.plot(freqs[mask], amp[mask])
ax.axvline(spindle, color='tab:red', linestyle='--', label='spindle')
ax.axvline(tpf, color='tab:blue', linestyle='--', label='tooth passing')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude')
ax.legend()
plt.show()